### AUTOCORRECTOR BASE + CONTEXTO

#### INDICE

- [Procedimiento basico](#Procedimiento-basico)
- [Evaluación de mejores hiperparametros](#Evaluación-de-mejores-hiperparametros)
- [Conclusiones](#Conclusiones)

### Procedimiento basico

<pre>
Autocorrector base + contexto (Procedimiento):
    1) Llega una frase a corregir.
    2) Para cada palabra de la frase:
        Si esta en el vocabulario => La dejamos igual
        Si no:
            Obtenemos varias palabras que están cerca, que son frecuentes y nos quedamos con la de mayor probabilidad
            de contexto.
    3) Se devuelve la frase corregida.
</pre>

### Evaluación de mejores hiperparametros

El autocorrector base + contexto usa 2 hiperpametros, el min_apperance, que es el número mínimo de apariciones
que debe tener una palabra para no ser considerada "unk"; y el tamaño de los ngramas.

En cuanto al min_apperance, vamos a probar todos los valores comprendidos entre [2, 20]. Sin embargo,
para el tamaño de los ngramas, solo va a ser de 2, es decir, no nos vamos a molestarnos en experimentar
con el porque en el caso de que sea 1, la información que aporta sobre el contexto es 0, y en el caso de
tener tamaño de ngrama > 2, no nos caben los datos en la memoria RAM.

In [1]:
from autocorrector import Autocorrector
from cargador_corpus import LectorCorpus

generador_muestras = LectorCorpus("../libros")
X_train = generador_muestras.get_corpus()
X_val = [
    ("el perro corria por el parque sin mirar atras", "el perro corria por el parqe sin mirar atras"),
    ("la niña jugaba tranquila en el jardin de su casa", "la niña jugaba tranquila en el jardn de su casa"),
    ("mañana iremos todos juntos a visitar a la abuela", "mañana iremos todos juntos a vsitar a la abuela"),
    ("el profesor explico la leccion con mucha paciencia", "el profesor explcio la leccion con mucha paciencia"),
    ("los amigos quedaron para cenar en un restaurante cercano", "los amigos quedaron para cenar en un restrante cercano"),
    ("la lluvia caia suavemente sobre el tejado de la casa", "la lluvia caia suavemente sobre el tejdao de la casa"),
    ("el tren llego puntual a la estacion principal del pueblo", "el tren llego puntual a la estcion principal del pueblo"),
    ("mi hermano pequeño rompio el juguete sin querer hacerlo", "mi hermano pequeño rompio el jugete sin querer hacerlo"),
    ("los niños aprendieron rapidamente a leer y escribir bien", "los niños aprendieron rapidamente a leer y escrbir bien"),
    ("el viento movia lentamente las hojas de los arboles", "el viento movia lentamente las hojas de los arbols"),
    ("la comida estaba deliciosa y todos repitieron varias veces", "la comida estaba delciosa y todos repitieron varias veces"),
    ("el coche azul estaba aparcado frente a la puerta principal", "el coche azul estaba aparcado frente a la pueta principal"),
    ("fuimos caminando despacio hasta llegar al final del sendero", "fuimos caminando despacio hasta llegar al fnal del sendero"),
    ("ella escribio una carta larga contando toda la verdad", "ella escribio una carta larga contando toda la vedad"),
    ("el gato dormia placidamente encima del sofa del salon", "el gato dormia placidamente encima del sfa del salon"),
    ("los estudiantes prepararon el examen durante toda la semana", "los estudiantes prepararon el exmen durante toda la semana"),
    ("la puerta vieja hacia ruido cada vez que alguien entraba", "la puerta vieja hacia ruido cada vez que algien entraba"),
    ("el jardin estaba lleno de flores de muchos colores distintos", "el jardin estaba lleno de flores de muchos colres distintos"),
    ("mi padre compro pan fresco en la tienda de la esquina", "mi padre compro pan fresco en la tenda de la esquina"),
    ("los vecinos hablaron durante horas sobre el problema comun", "los vecinos hablaron durante horas sobre el problma comun"),
]

print("=================================================")
print("    Evaluación de hiperparámetros")
print("=================================================\n")

best_acc = -1
best_min_apperance = -1
for min_apperance in range(2, 21):
    print(f"Evaluando para min_apperance={min_apperance}", end="")
    autocorrector_contexto = Autocorrector(modo="contexto", numero_ngramas=2, min_apperance=min_apperance)
    autocorrector_contexto.fit(X_train)

    acc = 0
    for frase_original, frase_con_error in X_val:
        frase_corregida = autocorrector_contexto.corregir(frase_con_error)
        if frase_corregida == frase_original:
            acc += 1
    acc /= len(X_val)
    print(f" acc = {acc}")

    if acc > best_acc:
        best_acc = acc
        best_min_apperance = min_apperance

    Evaluación de hiperparámetros

Evaluando para min_apperance=2 acc = 0.25
Evaluando para min_apperance=3 acc = 0.3
Evaluando para min_apperance=4 acc = 0.25
Evaluando para min_apperance=5 acc = 0.25
Evaluando para min_apperance=6 acc = 0.25
Evaluando para min_apperance=7 acc = 0.2
Evaluando para min_apperance=8 acc = 0.2
Evaluando para min_apperance=9 acc = 0.2
Evaluando para min_apperance=10 acc = 0.2
Evaluando para min_apperance=11 acc = 0.25
Evaluando para min_apperance=12 acc = 0.25
Evaluando para min_apperance=13 acc = 0.25
Evaluando para min_apperance=14 acc = 0.25
Evaluando para min_apperance=15 acc = 0.25
Evaluando para min_apperance=16 acc = 0.25
Evaluando para min_apperance=17 acc = 0.25
Evaluando para min_apperance=18 acc = 0.25
Evaluando para min_apperance=19 acc = 0.25
Evaluando para min_apperance=20 acc = 0.25


### Conclusiones

Como podemos observar el mejor accuracy obtenido es 0.4, sin embargo, hay varios valores de min_apperance
para los que ocurre esto, por ello, escogemos el valor más alto de estos porque ayudará a que nuestro
autocorrector puede generalizar mucho mejor las palabras, así que el mejor es min_apperance = 11.